In [1]:
# imports
from transformers import AutoProcessor, AutoModelForCausalLM
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, Dataset
import torch.nn.functional as F
import os
from tqdm import tqdm
import shutil
import random
import pandas as pd
from glob import glob
import numpy as np
import seaborn as sns
from PIL import Image, ImageFilter, ImageEnhance
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.image as mpimg
import cv2
from ultralytics import YOLO
import clip
import re
import copy

# ML / evaluation (fusion classifier)
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Utils
from collections import defaultdict
from typing import List, Dict, Tuple, Optional

In [2]:
# Use GPU if available
print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

True
Using device: cuda


In [3]:
clip_model, preprocess = clip.load("ViT-B/32", device=device)

In [4]:
yolo_model = YOLO("yolov8m.pt")

In [5]:
UCF_ROOT = Path(r"UCF Crime Dataset")
SPLIT = "Train"
NORMAL_CLASS = "NormalVideos"

split_dir = UCF_ROOT / SPLIT
print("UCF_ROOT exists:", UCF_ROOT.exists())
print("split_dir exists:", split_dir.exists())
print("classes:", [p.name for p in split_dir.iterdir() if p.is_dir()])

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

UCF_ROOT exists: True
split_dir exists: True
classes: ['Abuse', 'Arrest', 'Arson', 'Assault', 'Burglary', 'Explosion', 'Fighting', 'NormalVideos', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 'Stealing', 'Vandalism']


In [6]:
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}
_LAST_NUM = re.compile(r"_(\d+)$")  # Abuse001_x264_120 -> 120

def frame_index(p: Path) -> int:
    m = _LAST_NUM.search(p.stem)
    return int(m.group(1)) if m else 0

In [7]:
def list_videos(split_dir: Path):
    items = []
    class_dirs = sorted([d for d in split_dir.iterdir() if d.is_dir()])

    for class_dir in class_dirs:
        video_dirs = sorted([v for v in class_dir.iterdir() if v.is_dir()])
        for video_dir in video_dirs:
            frames = [p for p in video_dir.iterdir()
                      if p.is_file() and p.suffix.lower() in IMG_EXTS]
            if not frames:
                continue
            frames = sorted(frames, key=frame_index)

            items.append({
                "class_name": class_dir.name,
                "video_id": video_dir.name,
                "video_dir": video_dir,
                "frames": frames
            })
    return items

videos = list_videos(split_dir)
print("Total videos:", len(videos))
print("Example item:", videos[0]["class_name"], videos[0]["video_id"], "frames:", len(videos[0]["frames"]))


Total videos: 1610
Example item: Abuse Abuse001_x264 frames: 273


In [8]:
classes = sorted({v["class_name"] for v in videos})
label2idx = {c:i for i,c in enumerate(classes)}
idx2label = {i:c for c,i in label2idx.items()}

print("Classes:", classes)
print("label2idx:", label2idx)


Classes: ['Abuse', 'Arrest', 'Arson', 'Assault', 'Burglary', 'Explosion', 'Fighting', 'NormalVideos', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 'Stealing', 'Vandalism']
label2idx: {'Abuse': 0, 'Arrest': 1, 'Arson': 2, 'Assault': 3, 'Burglary': 4, 'Explosion': 5, 'Fighting': 6, 'NormalVideos': 7, 'RoadAccidents': 8, 'Robbery': 9, 'Shooting': 10, 'Shoplifting': 11, 'Stealing': 12, 'Vandalism': 13}


In [9]:
def uniform_sample_indices(n_frames: int, n_sample: int) -> np.ndarray:
    if n_frames <= 0:
        return np.array([], dtype=int)
    if n_frames >= n_sample:
        return np.linspace(0, n_frames - 1, num=n_sample, dtype=int)
    return np.linspace(0, n_frames - 1, num=n_sample, dtype=int) 


In [10]:
class UCFCrimeVideoDataset(Dataset):
    def __init__(self, video_items, label2idx, n_frames=32, transform=None,
                 return_binary=False, normal_class="NormalVideos"):
        self.video_items = video_items
        self.label2idx = label2idx
        self.n_frames = n_frames
        self.transform = transform
        self.return_binary = return_binary
        self.normal_class = normal_class

    def __len__(self):
        return len(self.video_items)

    def __getitem__(self, i):
        item = self.video_items[i]
        frames = item["frames"]
        idxs = uniform_sample_indices(len(frames), self.n_frames)

        imgs = []
        for k in idxs:
            img = Image.open(frames[int(k)]).convert("RGB")
            if self.transform is not None:
                img = self.transform(img)  # tensor
            imgs.append(img)

        if self.transform is not None:
            x = torch.stack(imgs, dim=0)  # [T,3,H,W]
        else:
            x = imgs  # list of PIL

        class_name = item["class_name"]
        y_multi = self.label2idx[class_name]
        y_bin = 0 if class_name == self.normal_class else 1

        if self.return_binary:
            return x, y_bin, y_multi, item["video_id"], class_name
        return x, y_multi, item["video_id"], class_name


In [11]:
ds = UCFCrimeVideoDataset(
    videos, label2idx,
    n_frames=32,
    transform=preprocess,        
    return_binary=True,
    normal_class=NORMAL_CLASS
)

dl = DataLoader(ds, batch_size=2, shuffle=True, num_workers=0)

In [12]:
@torch.no_grad()
def clip_batch_to_video_vectors(x, clip_model, device):
    """
    Convert a batch of videos (as sampled frames) into one fixed-size CLIP vector per video.

    x: [B, T, 3, 224, 224]
    returns: [B, 2D] where D is CLIP embedding dim (ViT-B/32 usually 512)
    """

    clip_model.eval()  # Set model to evaluation mode (no dropout, stable inference)

    # Unpack input dimensions
    B, T, C, H, W = x.shape

    # Move input to GPU/CPU device
    x = x.to(device)

    # Flatten videos into a single batch of frames:
    # [B, T, 3, 224, 224] -> [B*T, 3, 224, 224]
    xf = x.view(B * T, C, H, W)

    # Encode each frame with CLIP to get an embedding per frame:
    # [B*T, 3, 224, 224] -> [B*T, D]
    feats = clip_model.encode_image(xf)

    # Normalize embeddings to unit length (helps stability and similarity-based learning)
    feats = F.normalize(feats, dim=-1)

    # Reshape back to per-video sequence:
    # [B*T, D] -> [B, T, D]
    D = feats.shape[-1]
    feats = feats.view(B, T, D)

    # Pool over time to get one vector per video:
    # mean pooling captures "overall" content across frames
    v_mean = feats.mean(dim=1)            # [B, D]
    # max pooling captures the strongest signal across frames (useful if anomaly appears briefly)
    v_max  = feats.max(dim=1).values      # [B, D]

    # Concatenate mean and max to form the final video representation:
    # [B, D] + [B, D] -> [B, 2D]
    video_vecs = torch.cat([v_mean, v_max], dim=-1)

    return video_vecs

In [13]:
x, y_bin, y_multi, video_ids, class_names = next(iter(dl))  # Take one batch from DataLoader

video_vecs = clip_batch_to_video_vectors(x, clip_model, device)

print("x shape:", x.shape)                    # Expected: [B, T, 3, 224, 224]
print("video_vecs shape:", video_vecs.shape)  # Expected: [B, 2D] e.g., [2, 1024]
print("video_ids:", video_ids)                # Debug: which video folders were sampled
print("class_names:", class_names)            # Debug: their class names
print("binary:", y_bin)                       # 0=NormalVideos, 1=Anomaly
print("multi:", y_multi)                      # Multiclass label indices

x shape: torch.Size([2, 32, 3, 224, 224])
video_vecs shape: torch.Size([2, 1024])
video_ids: ('Normal_Videos489_x264', 'Robbery062_x264')
class_names: ('NormalVideos', 'Robbery')
binary: tensor([0, 1])
multi: tensor([7, 9])


In [14]:
@torch.no_grad()
def extract_clip_embeddings_from_loader(dl, clip_model, device):
    clip_model.eval()

    X_list = []
    ybin_list = []
    ymulti_list = []
    vid_list = []
    cname_list = []

    for x, y_bin, y_multi, video_ids, class_names in dl:
        # x: [B, T, 3, 224, 224]
        video_vecs = clip_batch_to_video_vectors(x, clip_model, device)  # [B, 2D]

        # Move to CPU for storage
        X_list.append(video_vecs.cpu())
        ybin_list.append(y_bin.cpu())
        ymulti_list.append(y_multi.cpu())

        # Keep ids/names as python lists
        vid_list.extend(list(video_ids))
        cname_list.extend(list(class_names))

    X_embed = torch.cat(X_list, dim=0)
    y_bin_all = torch.cat(ybin_list, dim=0)
    y_multi_all = torch.cat(ymulti_list, dim=0)

    return X_embed, y_bin_all, y_multi_all, vid_list, cname_list

In [15]:
X_embed, y_bin_all, y_multi_all, vid_list, cname_list = extract_clip_embeddings_from_loader(dl, clip_model, device)

print("X_embed shape:", X_embed.shape)
print("y_bin shape:", y_bin_all.shape, "| positives:", int((y_bin_all == 1).sum()))
print("y_multi shape:", y_multi_all.shape)
print("Example:", vid_list[0], cname_list[0], "bin=", int(y_bin_all[0]), "multi=", int(y_multi_all[0]))

X_embed shape: torch.Size([1610, 1024])
y_bin shape: torch.Size([1610]) | positives: 810
y_multi shape: torch.Size([1610])
Example: Normal_Videos707_x264 NormalVideos bin= 0 multi= 7


In [ ]:
# # save to disk (so you don't recompute embeddings every time)
# save_path = "ucf_clip_embeddings_train.pt"
# torch.save(
#     {
#         "X_embed": X_embed,
#         "y_bin": y_bin_all,
#         "y_multi": y_multi_all,
#         "video_ids": vid_list,
#         "class_names": cname_list,
#     },
#     save_path,
# )

# print("Saved:", save_path)

Saved: ucf_clip_embeddings_train.pt


In [16]:
data = torch.load("ucf_clip_embeddings_train.pt", map_location="cpu")

X_embed = data["X_embed"]          # [N, 1024]
y_bin_all = data["y_bin"]          # [N]
y_multi_all = data["y_multi"]      # [N]
vid_list = data["video_ids"]       # list[str]
cname_list = data["class_names"]   # list[str]

print("Loaded:")
print("X_embed:", X_embed.shape)
print("y_bin:", y_bin_all.shape, "| positives:", int((y_bin_all == 1).sum()))
print("y_multi:", y_multi_all.shape)
print("Example:", vid_list[0], cname_list[0], "bin=", int(y_bin_all[0]), "multi=", int(y_multi_all[0]))

Loaded:
X_embed: torch.Size([1610, 1024])
y_bin: torch.Size([1610]) | positives: 810
y_multi: torch.Size([1610])
Example: Arson053_x264 Arson bin= 1 multi= 2


In [17]:
# Convert to numpy for splitting
X_np = X_embed.numpy()
y_np = y_bin_all.numpy()

# Stratified split keeps the same anomaly ratio in train and val
X_train, X_val, y_train, y_val = train_test_split(
    X_np, y_np,
    test_size=0.2,
    random_state=42,
    stratify=y_np
)

print("Train:", X_train.shape, "positives:", int(y_train.sum()))
print("Val  :", X_val.shape,   "positives:", int(y_val.sum()))


Train: (1288, 1024) positives: 648
Val  : (322, 1024) positives: 162


In [18]:
# Convert to torch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)  # float for BCEWithLogitsLoss

X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

train_ds = TensorDataset(X_train_t, y_train_t)
val_ds = TensorDataset(X_val_t, y_val_t)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

print("train batches:", len(train_loader))
print("val batches:", len(val_loader))

train batches: 21
val batches: 2


In [19]:
class BinaryMLP(nn.Module):
    def __init__(self, in_dim=1024, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)  # one logit
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)  # [B]

model = BinaryMLP(in_dim=X_train_t.shape[1], hidden_dim=256, dropout=0.3).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)

print(model)

BinaryMLP(
  (net): Sequential(
    (0): Linear(in_features=1024, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=1, bias=True)
  )
)


In [20]:
def train_model(
    model, loader, optimizer, criterion, device,
    epochs=10,
    val_loader=None,
    patience=5,
    save_best_path="best_binary_mlp.pt"
):
    train_losses = []

    best_val_loss = float("inf")
    best_state = None
    bad_epochs = 0

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        pbar = tqdm(loader, total=len(loader), desc=f"Train Epoch {ep}/{epochs}", leave=False)

        for Xb, yb in pbar:
            Xb = Xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(Xb)           
            loss = criterion(logits, yb) 
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * Xb.size(0)

            # Update progress bar with current loss
            pbar.set_postfix(loss=float(loss.item()))

        avg_loss = total_loss / len(loader.dataset)
        train_losses.append(avg_loss)

        # Print epoch summary (same as you had)
        msg = f"Epoch {ep:02d}/{epochs} | train_loss={avg_loss:.4f}"

        if val_loader is not None:
            val_loss, _, _ = eval_model(model, val_loader, criterion, device)

            msg += f" | val_loss={val_loss:.4f}"

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = copy.deepcopy(model.state_dict())
                torch.save(best_state, save_best_path)
                bad_epochs = 0
            else:
                bad_epochs += 1
                msg += f" | patience {bad_epochs}/{patience}"

            print(msg)

            if bad_epochs >= patience:
                print(f"Early stopping: no val_loss improvement for {patience} epochs.")
                break
        else:
            print(msg)

    # ---- Added: load best weights back into the model ----
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"Loaded best weights | best_val_loss={best_val_loss:.4f} | saved to: {save_best_path}")

    return train_losses


@torch.no_grad()
def eval_model(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    all_probs = []
    all_y = []

    pbar = tqdm(loader, total=len(loader), desc="Eval", leave=False)

    for Xb, yb in pbar:
        Xb = Xb.to(device)
        yb = yb.to(device)

        logits = model(Xb)
        loss = criterion(logits, yb)

        probs = torch.sigmoid(logits)

        total_loss += loss.item() * Xb.size(0)
        all_probs.append(probs.cpu())
        all_y.append(yb.cpu())

        # Update progress bar with current loss
        pbar.set_postfix(loss=float(loss.item()))

    val_loss = total_loss / len(loader.dataset)
    all_probs = torch.cat(all_probs).numpy()
    all_y = torch.cat(all_y).numpy()

    print(f"Eval | val_loss={val_loss:.4f}")
    return val_loss, all_probs, all_y


In [21]:
train_losses = train_model(
    model, train_loader, optimizer, criterion, device,
    epochs=50,
    val_loader=val_loader,
    patience=7,
    save_best_path="best_binary_mlp.pt"
)

# model is already loaded with best weights
val_loss, val_probs, val_y = eval_model(model, val_loader, criterion, device)


Eval | val_loss=0.6572
Epoch 01/50 | train_loss=0.6787 | val_loss=0.6572


Eval | val_loss=0.6119
Epoch 02/50 | train_loss=0.6376 | val_loss=0.6119


Eval | val_loss=0.5595
Epoch 03/50 | train_loss=0.5886 | val_loss=0.5595


Eval | val_loss=0.5081
Epoch 04/50 | train_loss=0.5365 | val_loss=0.5081


Eval | val_loss=0.4661
Epoch 05/50 | train_loss=0.4899 | val_loss=0.4661


...

Eval | val_loss=0.3398
Epoch 38/50 | train_loss=0.2873 | val_loss=0.3398 | patience 4/7


Eval | val_loss=0.3399
Epoch 39/50 | train_loss=0.2831 | val_loss=0.3399 | patience 5/7


Eval | val_loss=0.3396
Epoch 40/50 | train_loss=0.2803 | val_loss=0.3396 | patience 6/7


Eval | val_loss=0.3391
Epoch 41/50 | train_loss=0.2774 | val_loss=0.3391 | patience 7/7
Early stopping: no val_loss improvement for 7 epochs.
Loaded best weights | best_val_loss=0.3387 | saved to: best_binary_mlp.pt


Eval | val_loss=0.3387


In [23]:
val_pred = (val_probs >= 0.4).astype(int)

acc = accuracy_score(val_y, val_pred)
f1 = f1_score(val_y, val_pred)
cm = confusion_matrix(val_y, val_pred)

print("Accuracy:", acc)
print("F1:", f1)
print("Confusion matrix:\n", cm)
print("\nReport:\n", classification_report(val_y, val_pred, digits=4))


Accuracy: 0.8633540372670807
F1: 0.8713450292397661
Confusion matrix:
 [[129  31]
 [ 13 149]]

Report:
               precision    recall  f1-score   support

         0.0     0.9085    0.8063    0.8543       160
         1.0     0.8278    0.9198    0.8713       162

    accuracy                         0.8634       322
   macro avg     0.8681    0.8630    0.8628       322
weighted avg     0.8679    0.8634    0.8629       322



In [165]:
torch.save(model.state_dict(), "binary_clip_mlp_best.pt")
print("Saved: binary_clip_mlp_best.pt")

Saved: binary_clip_mlp_best.pt


YOLO

In [91]:
# Build model skeleton
clip_trained_model = BinaryMLP(in_dim=1024, hidden_dim=256, dropout=0.2).to(device)

# Load weights (state_dict)
state = torch.load("binary_clip_mlp_best.pt", map_location="cpu")

# If the file is a full checkpoint dict, handle that too
if isinstance(state, dict) and "state_dict" in state:
    state = state["state_dict"]

clip_trained_model.load_state_dict(state)
clip_trained_model.eval()

print("Loaded BinaryMLP weights from: binary_clip_mlp_best.pt")
print("Model is ready on device:", device)


Loaded BinaryMLP weights from: binary_clip_mlp_best.pt
Model is ready on device: cuda


In [92]:
# X_embed_all: [1610, 1024] (from your saved embeddings file)
data = torch.load("ucf_clip_embeddings_train.pt", map_location="cpu")
X_embed_all = data["X_embed"]

with torch.no_grad():
    logits = clip_trained_model(X_embed_all[:8].to(device).float())
    probs = torch.sigmoid(logits).cpu()

print("probs:", probs.numpy())


probs: [    0.98386   0.0087723     0.71003     0.87514     0.89537     0.17376     0.97305    0.049853]


In [93]:
PERSON_ID = 0
VEHICLE_IDS = {1, 2, 3, 5, 7}  # bicycle, car, motorcycle, bus, truck

def sample_uniform(paths, n=32):
    """Return up to n paths sampled uniformly from the list."""
    if len(paths) <= n:
        return list(paths)
    idx = np.linspace(0, len(paths) - 1, n).round().astype(int)
    return [paths[i] for i in idx]

def yolo_video_features_from_paths(frame_paths, yolo_model, device, imgsz=640, conf=0.25):
    """
    Run YOLO on a list of frame image paths and return a fixed-size feature vector.

    Features (K=10):
      0 mean_people, 1 max_people
      2 mean_vehicles, 3 max_vehicles
      4 mean_total, 5 max_total
      6 mean_conf, 7 max_conf
      8 mean_box_area_ratio, 9 max_box_area_ratio
    """
    if len(frame_paths) == 0:
        return np.zeros(10, dtype=np.float32)

    results = yolo_model.predict(
        source=[str(p) for p in frame_paths],
        imgsz=imgsz,
        conf=conf,
        device=device,     # can be 0 for cuda:0, or "cpu"
        verbose=False
    )

    people_counts = []
    vehicle_counts = []
    total_counts = []
    conf_means = []
    area_ratios = []

    for r in results:
        boxes = r.boxes
        if boxes is None or len(boxes) == 0:
            people_counts.append(0)
            vehicle_counts.append(0)
            total_counts.append(0)
            conf_means.append(0.0)
            area_ratios.append(0.0)
            continue

        cls = boxes.cls.detach().cpu().numpy().astype(int)
        confs = boxes.conf.detach().cpu().numpy()
        xyxy = boxes.xyxy.detach().cpu().numpy()  # [N,4]

        people_counts.append(int((cls == PERSON_ID).sum()))
        vehicle_counts.append(int(np.isin(cls, list(VEHICLE_IDS)).sum()))
        total_counts.append(int(len(cls)))

        conf_means.append(float(confs.mean()) if len(confs) else 0.0)

        H, W = r.orig_shape
        areas = (xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])
        area_ratio = float(areas.sum() / (H * W + 1e-9))
        area_ratios.append(area_ratio)

    feat = np.array([
        float(np.mean(people_counts)),  float(np.max(people_counts)),
        float(np.mean(vehicle_counts)), float(np.max(vehicle_counts)),
        float(np.mean(total_counts)),   float(np.max(total_counts)),
        float(np.mean(conf_means)),     float(np.max(conf_means)),
        float(np.mean(area_ratios)),    float(np.max(area_ratios)),
    ], dtype=np.float32)

    return feat


In [94]:
v = videos[0]  # or any videos[i]
frame_paths = sample_uniform(v["frames"], n=32)

feat = yolo_video_features_from_paths(frame_paths, yolo_model, device=device)

print("video_id:", v["video_id"], "| class:", v["class_name"])
print("num sampled frames:", len(frame_paths))
print("YOLO feature vector (len={}):".format(len(feat)), feat)


video_id: Abuse001_x264 | class: Abuse
num sampled frames: 32
YOLO feature vector (len=10): [    0.09375           1           0           0     0.15625           1    0.053699     0.44466   0.0031876    0.029771]


In [32]:
X_yolo = []
yolo_vids = []
yolo_classes = []
yolo_ybin = []
yolo_ymulti = []

for v in tqdm(videos, desc="Extract YOLO features", total=len(videos)):
    frame_paths = sample_uniform(v["frames"], n=32)

    feat = yolo_video_features_from_paths(
        frame_paths,
        yolo_model,
        device=device,   # keep your existing device
        imgsz=640,
        conf=0.25
    )

    X_yolo.append(feat)
    yolo_vids.append(v["video_id"])
    yolo_classes.append(v["class_name"])

    yolo_ybin.append(0 if v["class_name"] == NORMAL_CLASS else 1)
    yolo_ymulti.append(label2idx[v["class_name"]])

X_yolo = torch.tensor(np.stack(X_yolo), dtype=torch.float32)

yolo_pack = {
    "X_yolo": X_yolo,                              # [N, 10]
    "video_ids": np.array(yolo_vids, dtype=object),
    "class_names": np.array(yolo_classes, dtype=object),
    "y_bin": torch.tensor(yolo_ybin, dtype=torch.long),
    "y_multi": torch.tensor(yolo_ymulti, dtype=torch.long),
}

torch.save(yolo_pack, "ucf_yolo_features_train.pt")
print("Saved: ucf_yolo_features_train.pt | X_yolo shape:", X_yolo.shape)


Extract YOLO features: 100%|██████████| 1610/1610 [21:44<00:00,  1.23it/s]

Saved: ucf_yolo_features_train.pt | X_yolo shape: torch.Size([1610, 10])


In [95]:
clip_data = torch.load("ucf_clip_embeddings_train.pt", map_location="cpu")
yolo_data = torch.load("ucf_yolo_features_train.pt", map_location="cpu")

print("Loaded CLIP keys:", clip_data.keys())
print("Loaded YOLO keys:", yolo_data.keys())


Loaded CLIP keys: dict_keys(['X_embed', 'y_bin', 'y_multi', 'video_ids', 'class_names'])
Loaded YOLO keys: dict_keys(['X_yolo', 'video_ids', 'class_names', 'y_bin', 'y_multi'])


In [96]:
# CLIP
X_embed_all = clip_data["X_embed"].float()       # [N, 1024]
y_bin_all   = clip_data["y_bin"].long()          # [N]
vid_all     = clip_data["video_ids"]             # (N,)

# YOLO
X_yolo_all  = yolo_data["X_yolo"].float()        # [N, 10]
vid_yolo    = yolo_data["video_ids"]             # (N,)

print("X_embed_all:", X_embed_all.shape)
print("y_bin_all  :", y_bin_all.shape)
print("X_yolo_all :", X_yolo_all.shape)
print("vid_all len:", len(vid_all))
print("vid_yolo len:", len(vid_yolo))

X_embed_all: torch.Size([1610, 1024])
y_bin_all  : torch.Size([1610])
X_yolo_all : torch.Size([1610, 10])
vid_all len: 1610
vid_yolo len: 1610


In [98]:
idx_yolo = {vid_yolo[i]: i for i in range(len(vid_yolo))}

yolo_aligned = []
missing = 0

for vid in vid_all:
    j = idx_yolo.get(vid, None)
    if j is None:
        missing += 1
        yolo_aligned.append(np.zeros((X_yolo_all.shape[1],), dtype=np.float32))
    else:
        yolo_aligned.append(X_yolo_all[j].numpy())

yolo_aligned = torch.tensor(np.stack(yolo_aligned), dtype=torch.float32)

print("yolo_aligned:", yolo_aligned.shape)
print("missing alignments:", missing)

yolo_aligned: torch.Size([1610, 10])
missing alignments: 0


In [99]:
X_fusion_all = torch.cat([X_embed_all, yolo_aligned], dim=1)  # [N, 1034]
print("X_fusion_all:", X_fusion_all.shape)


X_fusion_all: torch.Size([1610, 1034])


In [100]:
idx = np.arange(len(y_bin_all))

train_idx, val_idx = train_test_split(
    idx,
    test_size=0.2,
    random_state=42,
    stratify=y_bin_all.numpy()
)

print("train size:", len(train_idx))
print("val size  :", len(val_idx))

train size: 1288
val size  : 322


In [101]:
Xf_train = X_fusion_all[train_idx].float()
yf_train = y_bin_all[train_idx].float()   # float for BCEWithLogitsLoss

Xf_val   = X_fusion_all[val_idx].float()
yf_val   = y_bin_all[val_idx].float()

train_ds = TensorDataset(Xf_train, yf_train)
val_ds   = TensorDataset(Xf_val, yf_val)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=256, shuffle=False)

print("Fusion Train:", Xf_train.shape, "positives:", int((yf_train == 1).sum()))
print("Fusion Val  :", Xf_val.shape,   "positives:", int((yf_val == 1).sum()))

Fusion Train: torch.Size([1288, 1034]) positives: 648
Fusion Val  : torch.Size([322, 1034]) positives: 162


In [102]:
fusion_model = BinaryMLP(in_dim=X_fusion_all.shape[1], hidden_dim=256, dropout=0.2).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(fusion_model.parameters(), lr=3e-4, weight_decay=1e-4)

print(fusion_model)

BinaryMLP(
  (net): Sequential(
    (0): Linear(in_features=1034, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=256, out_features=1, bias=True)
  )
)


In [103]:
train_model(
    fusion_model,
    train_loader,
    optimizer,
    criterion,
    device,
    epochs=50,
    val_loader=val_loader,
    patience=7,
    save_best_path="best_fusion_binary_mlp.pt"
)

Eval | val_loss=0.6311
Epoch 01/50 | train_loss=0.6622 | val_loss=0.6311


Eval | val_loss=0.5742
Epoch 02/50 | train_loss=0.6067 | val_loss=0.5742


Eval | val_loss=0.5209
Epoch 03/50 | train_loss=0.5497 | val_loss=0.5209


Eval | val_loss=0.4773
Epoch 04/50 | train_loss=0.5025 | val_loss=0.4773


Eval | val_loss=0.4458
Epoch 05/50 | train_loss=0.4607 | val_loss=0.4458


...

Eval | val_loss=0.3531
Epoch 39/50 | train_loss=0.2712 | val_loss=0.3531 | patience 4/7


Eval | val_loss=0.3545
Epoch 40/50 | train_loss=0.2685 | val_loss=0.3545 | patience 5/7


Eval | val_loss=0.3521
Epoch 41/50 | train_loss=0.2654 | val_loss=0.3521 | patience 6/7


Eval | val_loss=0.3582
Epoch 42/50 | train_loss=0.2623 | val_loss=0.3582 | patience 7/7
Early stopping: no val_loss improvement for 7 epochs.
Loaded best weights | best_val_loss=0.3519 | saved to: best_fusion_binary_mlp.pt


[0.6621879383643962,
 0.6067360961659355,
 0.549742252930351,
 0.502453110973287,
 0.4607195826420873,
 0.4330820800354762,
 0.41607329915769353,
 0.39838546958769333,
 0.38659549388826264,
 0.375027430723913,
 0.3683779800530546,
 0.3645479605064629,
 0.35762801331392724,
 0.3488767130404526,
 0.3460740649737186,
 0.34155861117084574,
 0.33855761097084663,
 0.33400393568951153,
 0.3280957868750791,
 0.3298986313505943,
 0.321414499919607,
 0.32252518749385145,
 0.3180558583381013,
 0.31469586797012306,
 0.30917798288120246,
 0.3068881047808606,
 0.3024111356794464,
 0.2961689093098137,
 0.296942929303424,
 0.2976709537624572,
 0.2982893615775967,
 0.2868251121192245,
 0.2880704421434343,
 0.2868692330692125,
 0.27941976691254916,
 0.27642194977643325,
 0.2757755654013675,
 0.2742498421706028,
 0.2711979881021547,
 0.2685395971588466,
 0.26537287679518234,
 0.26228219344749215]

In [108]:
@torch.no_grad()
def predict_probs(model, X, device):
    model.eval()
    logits = model(X.to(device).float())
    return torch.sigmoid(logits).cpu().numpy()

# Fusion predictions on VAL
fusion_probs = predict_probs(fusion_model, Xf_val, device)
fusion_pred  = (fusion_probs >= 0.4).astype(int)
y_true       = yf_val.cpu().numpy().astype(int)

print("FUSION Accuracy:", accuracy_score(y_true, fusion_pred))
print("FUSION F1:", f1_score(y_true, fusion_pred))
print("FUSION Confusion:\n", confusion_matrix(y_true, fusion_pred))
print(classification_report(y_true, fusion_pred, digits=4))

FUSION Accuracy: 0.8571428571428571
FUSION F1: 0.8622754491017964
FUSION Confusion:
 [[132  28]
 [ 18 144]]
              precision    recall  f1-score   support

           0     0.8800    0.8250    0.8516       160
           1     0.8372    0.8889    0.8623       162

    accuracy                         0.8571       322
   macro avg     0.8586    0.8569    0.8569       322
weighted avg     0.8585    0.8571    0.8570       322



In [105]:
# CLIP-only baseline on SAME val split
Xclip_val = X_embed_all[val_idx].float()

clip_probs = predict_probs(clip_trained_model, Xclip_val, device)
clip_pred  = (clip_probs >= 0.4).astype(int)

print("\nCLIP-only Accuracy:", accuracy_score(y_true, clip_pred))
print("CLIP-only F1:", f1_score(y_true, clip_pred))
print("CLIP-only Confusion:\n", confusion_matrix(y_true, clip_pred))
print(classification_report(y_true, clip_pred, digits=4))



CLIP-only Accuracy: 0.8664596273291926
CLIP-only F1: 0.8731563421828908
CLIP-only Confusion:
 [[131  29]
 [ 14 148]]
              precision    recall  f1-score   support

           0     0.9034    0.8187    0.8590       160
           1     0.8362    0.9136    0.8732       162

    accuracy                         0.8665       322
   macro avg     0.8698    0.8662    0.8661       322
weighted avg     0.8696    0.8665    0.8661       322



## TESTING

In [80]:
# ---------------------- STEP 2 (One-Cell): Two-Tower + Gate Fusion ----------------------
# Assumes you already have in memory:
#   X_embed_all : torch.Tensor [N,1024]  (CLIP embeddings)
#   yolo_aligned: torch.Tensor [N,10]    (YOLO features aligned to CLIP video_ids)
#   y_bin_all   : torch.Tensor [N]       (0/1 labels)
#   train_idx, val_idx : arrays of indices from your split
#   clip_trained_model : your trained CLIP-only BinaryMLP model (loaded & on device)
#   device : torch.device("cuda" or "cpu")
#
# This cell trains Two-Tower+Gate fusion and compares vs CLIP-only on the SAME val split.

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm
import copy
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# ---------------------- Safety checks ----------------------
assert isinstance(X_embed_all, torch.Tensor) and X_embed_all.ndim == 2 and X_embed_all.shape[1] == 1024, \
    f"X_embed_all must be torch.Tensor [N,1024], got {type(X_embed_all)} {getattr(X_embed_all,'shape',None)}"
assert isinstance(yolo_aligned, torch.Tensor) and yolo_aligned.ndim == 2 and yolo_aligned.shape[1] == 10, \
    f"yolo_aligned must be torch.Tensor [N,10], got {type(yolo_aligned)} {getattr(yolo_aligned,'shape',None)}"
assert isinstance(y_bin_all, torch.Tensor) and y_bin_all.ndim == 1, \
    f"y_bin_all must be torch.Tensor [N], got {type(y_bin_all)} {getattr(y_bin_all,'shape',None)}"
assert len(train_idx) + len(val_idx) <= X_embed_all.shape[0] + 5, "train_idx/val_idx look wrong"
print("OK shapes:", X_embed_all.shape, yolo_aligned.shape, y_bin_all.shape, "| train/val:", len(train_idx), len(val_idx))

# ---------------------- Build loaders (Two-Tower inputs) ----------------------
Xc_train = X_embed_all[train_idx].float()
Xy_train = yolo_aligned[train_idx].float()
yf_train = y_bin_all[train_idx].float()

Xc_val   = X_embed_all[val_idx].float()
Xy_val   = yolo_aligned[val_idx].float()
yf_val   = y_bin_all[val_idx].float()

train_ds_tt = TensorDataset(Xc_train, Xy_train, yf_train)
val_ds_tt   = TensorDataset(Xc_val,   Xy_val,   yf_val)

train_loader_tt = DataLoader(train_ds_tt, batch_size=64, shuffle=True)
val_loader_tt   = DataLoader(val_ds_tt,   batch_size=256, shuffle=False)

print("TwoTower Train:", Xc_train.shape, Xy_train.shape, "positives:", int((yf_train == 1).sum()))
print("TwoTower Val  :", Xc_val.shape,   Xy_val.shape,   "positives:", int((yf_val == 1).sum()))

# ---------------------- Model: Two-Tower + Gate ----------------------
class FusionGatedMLP(nn.Module):
    def __init__(self, clip_dim=1024, yolo_dim=10, clip_h=256, yolo_h=64, fused_h=128, dropout=0.2):
        super().__init__()
        self.clip_ln = nn.LayerNorm(clip_dim)
        self.yolo_ln = nn.LayerNorm(yolo_dim)

        self.clip_net = nn.Sequential(
            nn.Linear(clip_dim, clip_h),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.yolo_net = nn.Sequential(
            nn.Linear(yolo_dim, yolo_h),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.gate = nn.Sequential(
            nn.Linear(clip_h + yolo_h, 1),
            nn.Sigmoid()
        )

        self.head = nn.Sequential(
            nn.Linear(clip_h + yolo_h, fused_h),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fused_h, 1)
        )

    def forward(self, x_clip, x_yolo):
        x_clip = self.clip_ln(x_clip)
        x_yolo = self.yolo_ln(x_yolo)

        c = self.clip_net(x_clip)               # [B, clip_h]
        y = self.yolo_net(x_yolo)               # [B, yolo_h]

        g = self.gate(torch.cat([c, y], dim=1)) # [B, 1]
        y = y * g                               # gated YOLO

        z = torch.cat([c, y], dim=1)
        return self.head(z).squeeze(-1)         # [B]

fusion_gated_model = FusionGatedMLP(
    clip_dim=1024, yolo_dim=10,
    clip_h=256, yolo_h=64,
    fused_h=128,
    dropout=0.2
).to(device)

# ---------------------- Train/Eval functions ----------------------
@torch.no_grad()
def eval_model_two_tower(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_y = []

    pbar = tqdm(loader, total=len(loader), desc="Eval", leave=False)
    for Xc, Xy, yb in pbar:
        Xc = Xc.to(device).float()
        Xy = Xy.to(device).float()
        yb = yb.to(device).float()

        logits = model(Xc, Xy)
        loss = criterion(logits, yb)
        probs = torch.sigmoid(logits)

        total_loss += loss.item() * yb.size(0)
        all_probs.append(probs.cpu())
        all_y.append(yb.cpu())

        pbar.set_postfix(loss=float(loss.item()))

    val_loss = total_loss / len(loader.dataset)
    all_probs = torch.cat(all_probs).numpy()
    all_y = torch.cat(all_y).numpy()

    print(f"Eval | val_loss={val_loss:.4f}")
    return val_loss, all_probs, all_y


def train_model_two_tower(
    model, loader, optimizer, criterion, device,
    epochs=50,
    val_loader=None,
    patience=7,
    save_best_path="best_fusion_gated_mlp.pt"
):
    train_losses = []
    best_val_loss = float("inf")
    best_state = None
    bad_epochs = 0

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        pbar = tqdm(loader, total=len(loader), desc=f"Train Epoch {ep}/{epochs}", leave=False)
        for Xc, Xy, yb in pbar:
            Xc = Xc.to(device).float()
            Xy = Xy.to(device).float()
            yb = yb.to(device).float()

            optimizer.zero_grad()
            logits = model(Xc, Xy)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * yb.size(0)
            pbar.set_postfix(loss=float(loss.item()))

        avg_loss = total_loss / len(loader.dataset)
        train_losses.append(avg_loss)

        msg = f"Epoch {ep:02d}/{epochs} | train_loss={avg_loss:.4f}"

        if val_loader is not None:
            val_loss, _, _ = eval_model_two_tower(model, val_loader, criterion, device)
            msg += f" | val_loss={val_loss:.4f}"

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = copy.deepcopy(model.state_dict())
                torch.save(best_state, save_best_path)
                bad_epochs = 0
            else:
                bad_epochs += 1
                msg += f" | patience {bad_epochs}/{patience}"

            print(msg)

            if bad_epochs >= patience:
                print(f"Early stopping: no val_loss improvement for {patience} epochs.")
                break
        else:
            print(msg)

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"Loaded best weights | best_val_loss={best_val_loss:.4f} | saved to: {save_best_path}")

    return train_losses


# ---------------------- Train ----------------------
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(fusion_gated_model.parameters(), lr=3e-4, weight_decay=1e-4)

_ = train_model_two_tower(
    fusion_gated_model,
    train_loader_tt,
    optimizer,
    criterion,
    device,
    epochs=50,
    val_loader=val_loader_tt,
    patience=7,
    save_best_path="best_fusion_gated_mlp.pt"
)

# ---------------------- Predict utilities ----------------------
@torch.no_grad()
def predict_probs_two_tower(model, Xc, Xy, device):
    model.eval()
    logits = model(Xc.to(device).float(), Xy.to(device).float())
    return torch.sigmoid(logits).cpu().numpy()

@torch.no_grad()
def predict_probs_clip_only(model, X, device):
    model.eval()
    logits = model(X.to(device).float())
    return torch.sigmoid(logits).cpu().numpy()

# ---------------------- Evaluate (same threshold you used before) ----------------------
THRESH = 0.4

y_true = yf_val.cpu().numpy().astype(int)

# Two-Tower Fusion
fusion_probs = predict_probs_two_tower(fusion_gated_model, Xc_val, Xy_val, device)
fusion_pred  = (fusion_probs >= THRESH).astype(int)

print("\nTWO-TOWER FUSION Accuracy:", accuracy_score(y_true, fusion_pred))
print("TWO-TOWER FUSION F1:", f1_score(y_true, fusion_pred))
print("TWO-TOWER FUSION Confusion:\n", confusion_matrix(y_true, fusion_pred))
print(classification_report(y_true, fusion_pred, digits=4))

# CLIP-only baseline (same val split)
clip_probs = predict_probs_clip_only(clip_trained_model, Xc_val, device)
clip_pred  = (clip_probs >= THRESH).astype(int)

print("\nCLIP-only Accuracy:", accuracy_score(y_true, clip_pred))
print("CLIP-only F1:", f1_score(y_true, clip_pred))
print("CLIP-only Confusion:\n", confusion_matrix(y_true, clip_pred))
print(classification_report(y_true, clip_pred, digits=4))
# --------------------------------------------------------------------------------------


OK shapes: torch.Size([1610, 1024]) torch.Size([1610, 10]) torch.Size([1610]) | train/val: 1288 322
TwoTower Train: torch.Size([1288, 1024]) torch.Size([1288, 10]) positives: 648
TwoTower Val  : torch.Size([322, 1024]) torch.Size([322, 10]) positives: 162


Eval | val_loss=0.4556
Epoch 01/50 | train_loss=0.5860 | val_loss=0.4556


Eval | val_loss=0.3759
Epoch 02/50 | train_loss=0.4054 | val_loss=0.3759


Eval | val_loss=0.3540
Epoch 03/50 | train_loss=0.3718 | val_loss=0.3540


Eval | val_loss=0.3425
Epoch 04/50 | train_loss=0.3453 | val_loss=0.3425


...

Eval | val_loss=0.3539
Epoch 11/50 | train_loss=0.2124 | val_loss=0.3539 | patience 3/7


Eval | val_loss=0.3923
Epoch 12/50 | train_loss=0.2064 | val_loss=0.3923 | patience 4/7


Eval | val_loss=0.3922
Epoch 13/50 | train_loss=0.1953 | val_loss=0.3922 | patience 5/7


Eval | val_loss=0.3748
Epoch 14/50 | train_loss=0.1818 | val_loss=0.3748 | patience 6/7


Eval | val_loss=0.4336
Epoch 15/50 | train_loss=0.1714 | val_loss=0.4336 | patience 7/7
Early stopping: no val_loss improvement for 7 epochs.
Loaded best weights | best_val_loss=0.3301 | saved to: best_fusion_gated_mlp.pt

TWO-TOWER FUSION Accuracy: 0.8726708074534162
TWO-TOWER FUSION F1: 0.8738461538461538
TWO-TOWER FUSION Confusion:
 [[139  21]
 [ 20 142]]
              precision    recall  f1-score   support

           0     0.8742    0.8688    0.8715       160
           1     0.8712    0.8765    0.8738       162

    accuracy                         0.8727       322
   macro avg     0.8727    0.8726    0.8727       322
weighted avg     0.8727    0.8727    0.8727       322


CLIP-only Accuracy: 0.8664596273291926
CLIP-only F1: 0.8731563421828908
CLIP-only Confusion:
 [[131  29]
 [ 14 148]]
              precision    recall  f1-score   support

           0     0.9034    0.8187    0.8590       160
           1     0.8362    0.9136    0.8732       162

    accuracy                  

In [89]:
# ---------------------- ONE CELL: Residual Fusion (CLIP logit + learned YOLO correction) ----------------------
# Goal: keep CLIP as the strong baseline, and let YOLO learn ONLY a correction (delta) on top of CLIP.
# This usually improves stability and prevents fusion from getting worse than CLIP.

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm
import numpy as np
import copy
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# ---------------------- Assumptions ----------------------
# You already have:
#   X_embed_all  : torch.Tensor [N,1024]  (CLIP embeddings)
#   yolo_aligned : torch.Tensor [N,10]    (YOLO features aligned)
#   y_bin_all    : torch.Tensor [N]       (0/1)
#   train_idx, val_idx
#   clip_trained_model : trained BinaryMLP for CLIP-only (outputs logits)
#   device

assert X_embed_all.shape[0] == yolo_aligned.shape[0] == y_bin_all.shape[0]
print("OK shapes:", X_embed_all.shape, yolo_aligned.shape, y_bin_all.shape, "| train/val:", len(train_idx), len(val_idx))

# ---------------------- Make sure CLIP model is float32 on device ----------------------
clip_trained_model = clip_trained_model.to(device).float()
clip_trained_model.eval()

# ---------------------- Precompute CLIP logits for ALL videos (fast, N=1610) ----------------------
@torch.no_grad()
def compute_clip_logits_all(clip_model, X_embed, device, bs=256):
    clip_model.eval()
    outs = []
    for i in range(0, X_embed.shape[0], bs):
        xb = X_embed[i:i+bs].to(device).float()
        logits = clip_model(xb)  # [B]
        outs.append(logits.detach().cpu())
    return torch.cat(outs, dim=0)  # [N]

clip_logits_all = compute_clip_logits_all(clip_trained_model, X_embed_all, device=device, bs=256)  # [N]
print("clip_logits_all:", clip_logits_all.shape)

# ---------------------- Split ----------------------
Xc_train = X_embed_all[train_idx].float()
Xy_train = yolo_aligned[train_idx].float()
yb_train = y_bin_all[train_idx].float()
cl_train = clip_logits_all[train_idx].float()

Xc_val   = X_embed_all[val_idx].float()
Xy_val   = yolo_aligned[val_idx].float()
yb_val   = y_bin_all[val_idx].float()
cl_val   = clip_logits_all[val_idx].float()

# ---------------------- Standardize YOLO using TRAIN stats only ----------------------
y_mean = Xy_train.mean(dim=0, keepdim=True)
y_std  = Xy_train.std(dim=0, keepdim=True).clamp_min(1e-6)

Xy_train_z = (Xy_train - y_mean) / y_std
Xy_val_z   = (Xy_val   - y_mean) / y_std

print("ResidualFusion Train:", Xc_train.shape, Xy_train_z.shape, "positives:", int((yb_train == 1).sum()))
print("ResidualFusion Val  :", Xc_val.shape,   Xy_val_z.shape,   "positives:", int((yb_val == 1).sum()))

train_ds = TensorDataset(Xc_train, Xy_train_z, cl_train, yb_train)
val_ds   = TensorDataset(Xc_val,   Xy_val_z,   cl_val,   yb_val)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False)

# ---------------------- Model: residual correction ----------------------
class ResidualFusionMLP(nn.Module):
    """
    output_logit = clip_logit + delta(clip_embed, yolo_feats)
    """
    def __init__(self, clip_dim=1024, yolo_dim=10, clip_h=256, yolo_h=64, fused_h=128, dropout=0.2):
        super().__init__()

        self.clip_ln = nn.LayerNorm(clip_dim)
        self.yolo_ln = nn.LayerNorm(yolo_dim)

        self.clip_proj = nn.Sequential(
            nn.Linear(clip_dim, clip_h),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.yolo_proj = nn.Sequential(
            nn.Linear(yolo_dim, yolo_h),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # A small gate so YOLO doesn't dominate
        self.gate = nn.Sequential(
            nn.Linear(clip_h + yolo_h, 1),
            nn.Sigmoid()
        )

        self.delta_head = nn.Sequential(
            nn.Linear(clip_h + yolo_h, fused_h),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fused_h, 1)
        )

    def forward(self, x_clip, x_yolo, clip_logit):
        x_clip = self.clip_ln(x_clip)
        x_yolo = self.yolo_ln(x_yolo)

        c = self.clip_proj(x_clip)  # [B,clip_h]
        y = self.yolo_proj(x_yolo)  # [B,yolo_h]

        g = self.gate(torch.cat([c, y], dim=1))  # [B,1]
        y = y * g

        delta = self.delta_head(torch.cat([c, y], dim=1)).squeeze(-1)  # [B]
        out_logit = clip_logit + delta
        return out_logit

fusion_model = ResidualFusionMLP(
    clip_dim=1024, yolo_dim=10,
    clip_h=256, yolo_h=64,
    fused_h=128, dropout=0.2
).to(device)

# ---------------------- Train / Eval (early stop by val F1, not loss) ----------------------
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(fusion_model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2, verbose=False)

@torch.no_grad()
def eval_residual(model, loader, criterion, device, thresh=0.4):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_y = []

    pbar = tqdm(loader, total=len(loader), desc="Eval", leave=False)
    for Xc, Xy, cl, yb in pbar:
        Xc = Xc.to(device).float()
        Xy = Xy.to(device).float()
        cl = cl.to(device).float()
        yb = yb.to(device).float()

        logits = model(Xc, Xy, cl)
        loss = criterion(logits, yb)
        probs = torch.sigmoid(logits)

        total_loss += loss.item() * yb.size(0)
        all_probs.append(probs.detach().cpu())
        all_y.append(yb.detach().cpu())

        pbar.set_postfix(loss=float(loss.item()))

    val_loss = total_loss / len(loader.dataset)
    probs = torch.cat(all_probs).numpy()
    y_true = torch.cat(all_y).numpy().astype(int)

    pred = (probs >= thresh).astype(int)
    val_f1 = f1_score(y_true, pred)
    val_acc = accuracy_score(y_true, pred)

    print(f"Eval | val_loss={val_loss:.4f} | val_acc@{thresh}={val_acc:.4f} | val_f1@{thresh}={val_f1:.4f}")
    return val_loss, probs, y_true, val_acc, val_f1

def train_residual(
    model, train_loader, val_loader, optimizer, criterion, device,
    epochs=50, patience=7, save_best_path="best_residual_fusion.pt", thresh=0.4
):
    best_f1 = -1.0
    best_state = None
    bad_epochs = 0
    train_losses = []

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        pbar = tqdm(train_loader, total=len(train_loader), desc=f"Train Epoch {ep}/{epochs}", leave=False)
        for Xc, Xy, cl, yb in pbar:
            Xc = Xc.to(device).float()
            Xy = Xy.to(device).float()
            cl = cl.to(device).float()
            yb = yb.to(device).float()

            optimizer.zero_grad()
            logits = model(Xc, Xy, cl)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * yb.size(0)
            pbar.set_postfix(loss=float(loss.item()))

        avg_loss = total_loss / len(train_loader.dataset)
        train_losses.append(avg_loss)

        # Validate
        val_loss, _, _, _, val_f1 = eval_residual(model, val_loader, criterion, device, thresh=thresh)
        scheduler.step(val_loss)

        msg = f"Epoch {ep:02d}/{epochs} | train_loss={avg_loss:.4f} | val_loss={val_loss:.4f} | val_f1@{thresh}={val_f1:.4f}"

        if val_f1 > best_f1 + 1e-6:
            best_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, save_best_path)
            bad_epochs = 0
        else:
            bad_epochs += 1
            msg += f" | patience {bad_epochs}/{patience}"

        print(msg)

        if bad_epochs >= patience:
            print(f"Early stopping: no val_F1 improvement for {patience} epochs.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"Loaded best weights | best_val_F1={best_f1:.4f} | saved to: {save_best_path}")

    return train_losses

# ---------------------- Train ----------------------
_ = train_residual(
    fusion_model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    epochs=50,
    patience=7,
    save_best_path="best_residual_fusion.pt",
    thresh=0.4
)

# ---------------------- Final evaluation @ fixed threshold ----------------------
val_loss, fusion_probs, y_true, acc_t, f1_t = eval_residual(fusion_model, val_loader, criterion, device, thresh=0.4)

pred = (fusion_probs >= 0.4).astype(int)
print("\nRESIDUAL FUSION @THRESH 0.4")
print("Accuracy:", accuracy_score(y_true, pred))
print("F1:", f1_score(y_true, pred))
print("Confusion:\n", confusion_matrix(y_true, pred))
print(classification_report(y_true, pred, digits=4))

# ---------------------- Compare to CLIP-only on SAME val split ----------------------
@torch.no_grad()
def clip_probs_on_val(clip_model, Xc_val, device):
    clip_model.eval()
    logits = clip_model(Xc_val.to(device).float())
    return torch.sigmoid(logits).detach().cpu().numpy()

clip_probs = clip_probs_on_val(clip_trained_model, Xc_val, device)
clip_pred  = (clip_probs >= 0.4).astype(int)

print("\nCLIP-only @THRESH 0.4")
print("Accuracy:", accuracy_score(y_true, clip_pred))
print("F1:", f1_score(y_true, clip_pred))
print("Confusion:\n", confusion_matrix(y_true, clip_pred))
print(classification_report(y_true, clip_pred, digits=4))

# ---------------------- Threshold scan (best F1) ----------------------
def scan_thresholds(probs, y_true, name="model"):
    best = None
    for t in np.linspace(0.05, 0.95, 91):
        pred = (probs >= t).astype(int)
        f1 = f1_score(y_true, pred)
        acc = accuracy_score(y_true, pred)
        cm = confusion_matrix(y_true, pred)
        if (best is None) or (f1 > best["f1"]):
            best = {"t": float(t), "f1": float(f1), "acc": float(acc), "cm": cm}
    print(f"\n{name} BEST by F1 -> threshold={best['t']:.2f} | F1={best['f1']:.4f} | Acc={best['acc']:.4f}")
    print("Confusion:\n", best["cm"])
    return best

best_res = scan_thresholds(fusion_probs, y_true, "RESIDUAL FUSION")
best_clip = scan_thresholds(clip_probs,   y_true, "CLIP-only")


OK shapes: torch.Size([1610, 1024]) torch.Size([1610, 10]) torch.Size([1610]) | train/val: 1288 322
clip_logits_all: torch.Size([1610])
ResidualFusion Train: torch.Size([1288, 1024]) torch.Size([1288, 10]) positives: 648
ResidualFusion Val  : torch.Size([322, 1024]) torch.Size([322, 10]) positives: 162


Eval | val_loss=0.3371 | val_acc@0.4=0.8602 | val_f1@0.4=0.8688
Epoch 01/50 | train_loss=0.2809 | val_loss=0.3371 | val_f1@0.4=0.8688


Eval | val_loss=0.3394 | val_acc@0.4=0.8696 | val_f1@0.4=0.8727


Epoch 02/50 | train_loss=0.2783 | val_loss=0.3394 | val_f1@0.4=0.8727


Eval | val_loss=0.3370 | val_acc@0.4=0.8571 | val_f1@0.4=0.8663
Epoch 03/50 | train_loss=0.2792 | val_loss=0.3370 | val_f1@0.4=0.8663 | patience 1/7


...

Eval | val_loss=0.3579 | val_acc@0.4=0.8727 | val_f1@0.4=0.8738
Epoch 16/50 | train_loss=0.1411 | val_loss=0.3579 | val_f1@0.4=0.8738 | patience 4/7


Eval | val_loss=0.3583 | val_acc@0.4=0.8540 | val_f1@0.4=0.8597
Epoch 17/50 | train_loss=0.1276 | val_loss=0.3583 | val_f1@0.4=0.8597 | patience 5/7


Eval | val_loss=0.3805 | val_acc@0.4=0.8385 | val_f1@0.4=0.8523
Epoch 18/50 | train_loss=0.1156 | val_loss=0.3805 | val_f1@0.4=0.8523 | patience 6/7


Eval | val_loss=0.3634 | val_acc@0.4=0.8571 | val_f1@0.4=0.8614
Epoch 19/50 | train_loss=0.1173 | val_loss=0.3634 | val_f1@0.4=0.8614 | patience 7/7
Early stopping: no val_F1 improvement for 7 epochs.
Loaded best weights | best_val_F1=0.8783 | saved to: best_residual_fusion.pt


Eval | val_loss=0.3276 | val_acc@0.4=0.8727 | val_f1@0.4=0.8783

RESIDUAL FUSION @THRESH 0.4
Accuracy: 0.8726708074534162
F1: 0.8783382789317508
Confusion:
 [[133  27]
 [ 14 148]]
              precision    recall  f1-score   support

           0     0.9048    0.8313    0.8664       160
           1     0.8457    0.9136    0.8783       162

    accuracy                         0.8727       322
   macro avg     0.8752    0.8724    0.8724       322
weighted avg     0.8751    0.8727    0.8724       322


CLIP-only @THRESH 0.4
Accuracy: 0.8664596273291926
F1: 0.8731563421828908
Confusion:
 [[131  29]
 [ 14 148]]
              precision    recall  f1-score   support

           0     0.9034    0.8187    0.8590       160
           1     0.8362    0.9136    0.8732       162

    accuracy                         0.8665       322
   macro avg     0.8698    0.8662    0.8661       322
weighted avg     0.8696    0.8665    0.8661       322


RESIDUAL FUSION BEST by F1 -> threshold=0.39 | F1=0.878